In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive/', force_remount=True)

os.chdir('drive/MyDrive/ExoVision')

In [ ]:
pip install -r requirements.txt

In [1]:
import sys
import os
import time
from lightkurve import search_targetpixelfile
from lightkurve import TessTargetPixelFile
import lightkurve as lk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from keras.models import load_model
from keras.optimizers import Adam
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Conv1D, MaxPooling1D, Flatten
from wotan import slide_clip
from wotan import transit_mask, flatten
from astropy.stats import sigma_clip
from astropy import units as u
import csv
import shutil
from scipy.interpolate import interp1d
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import make_forecasting_frame
from tsfresh.utilities.dataframe_functions import impute
from statsmodels.tsa.seasonal import seasonal_decompose
from multiprocessing import Pool
import multiprocessing
import numpy as np
import pandas as pd
import lightkurve as lk
from scipy.signal import find_peaks
from astropy.timeseries import BoxLeastSquares
from scipy.interpolate import interp1d

/Users/nasvinnabeel/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
%matplotlib inline

In [3]:
value_df = 10000
cwd = os.getcwd()
dirname = cwd[len(cwd)-len("Satellite Datasets"):len(cwd)]
if(dirname != 'Satellite Datasets'):
    os.chdir('./Satellite Datasets')
# datasets = os.listdir()
# stars1 = pd.read_csv('star1.csv')
# stars2 = pd.read_csv('star2.csv')
# stars3 = pd.read_csv('star3.csv')
# stars4 = pd.read_csv('star4.csv')
# stars = pd.concat([stars1,stars2,stars3,stars4])
# stars = stars.drop_duplicates('target_name')
# stars['target_name'] = stars['target_name'].apply(lambda x: 'TIC ' + str(x))
# confirmed_stars = pd.read_csv('confirmed_stars.csv')
# confirmed_stars['tid'] = confirmed_stars['tid'].apply(lambda x: 'TIC ' + str(x))
# stars['confirmed_planet'] = stars['target_name'].isin(confirmed_stars['tid']).astype(int)
# stars = stars.reset_index()
# confirmed_index = np.array(stars[stars['confirmed_planet']==1].index)
# unconfirmed_index = np.array(stars[stars['confirmed_planet']==0].index)

In [ ]:
# with open('exoplanet_star_updated_flux.csv', 'w', newline="", encoding="utf-8") as file:
#     csvwriter = csv.writer(file)
#     csvwriter.writerow(np.concatenate([['tid', 'confirmed_planet'], np.arange(1, 10001)]))

In [ ]:
# index_num = 7235
# for planet_index in range(5000,7000):
#     print('Starting index : ', planet_index)
#     for conf_index in ['confirmed','nonconfirmed']:
#         idno = ""
#         planet = ""
#         if conf_index == 'confirmed':
#             idno = stars.iloc[confirmed_index[planet_index]]['target_name']
#             planet = stars.iloc[confirmed_index[planet_index]]['confirmed_planet']
#         else:
#             idno = stars.iloc[unconfirmed_index[planet_index]]['target_name']
#             planet = stars.iloc[unconfirmed_index[planet_index]]['confirmed_planet']
#         sectorsdata = lk.search_lightcurve(idno, author=["TESS-SPOC"], exptime=1800)
#         if(sectorsdata.download_all()!= None):
#             sectors = sectorsdata.download_all()
#             shutil.rmtree('/Users/nasvinnabeel/.lightkurve/cache/mastDownload/')
#             for i in sectors:
#                 i.flux = i.pdcsap_flux.value.unmasked
#                 i.flux_err = i.pdcsap_flux_err.value.unmasked
#             stitched_lc = sectors.stitch()
#             time = stitched_lc.time.value
#             flux = stitched_lc.flux.value
#             min_period = 1
#             max_period = 1000
#             num_periods = 10000
#             period_time = np.logspace(np.log10(min_period), np.log10(max_period), num_periods)
#             bls_periodogram = stitched_lc.to_periodogram(method='bls', period=period_time)
#             planet_period = bls_periodogram.period_at_max_power
#             planet_t0 = bls_periodogram.transit_time_at_max_power
#             planet_duration = bls_periodogram.duration_at_max_power
#             folded_light_curve = stitched_lc.fold(period=planet_period, epoch_time=planet_t0)
#             flatten_lc, trend_lc = flatten(folded_light_curve.time.value, folded_light_curve.flux, method='biweight', return_trend=True)
#             light = pd.DataFrame({'Time':folded_light_curve.time.value,'Flux':flatten_lc}).dropna()
#             flux_series = pd.Series([i[0] for i in light[['Flux']].to_numpy()], index=[i[0] for i in light[['Time']].to_numpy()])
#             decompose_result_mult = seasonal_decompose(flux_series, model="additive", period=int(planet_period.value))
#             trend = decompose_result_mult.trend
#             if trend.shape[0] < 10000:
#                 arr_val = np.full(value_df, pd.DataFrame(trend).mean()[0])
#                 arr_val[5000-(trend.shape[0]//2):5000+(trend.shape[0]//2)] = trend.to_numpy()[:2*(trend.shape[0]//2)]
#             else:
#                 arr_val = trend.to_numpy()[(trend.shape[0]//2-5000):(trend.shape[0]//2+5000)]
#             with open('exoplanet_star_updated_flux.csv', 'a', newline="", encoding="utf-8") as file:
#                 csvwriter = csv.writer(file)
#                 st = np.concatenate([[idno,planet],arr_val])
#                 csvwriter.writerow(st)
#             index_num = index_num + 1
#             print("Planets Added to the database",index_num)
#     print("Succesfully Finished Adding Planet Index",planet_index,)

In [4]:
star = pd.read_csv('exoplanet_star_updated_flux.csv')

In [9]:
star

,tid,confirmed_planet,1,2,3,4,5,6,7,8,...,9991,9992,9993,9994,9995,9996,9997,9998,9999,10000
0,TIC 18067025,1,0.999847,0.999847,0.999847,0.999847,0.999847,0.999847,0.999847,0.999847,...,0.999847,0.999847,0.999847,0.999847,0.999847,0.999847,0.999847,0.999847,0.999847,0.999847
1,TIC 4749723,0,1.000002,1.000002,1.000002,1.000002,1.000002,1.000002,1.000002,1.000002,...,1.000002,1.000002,1.000002,1.000002,1.000002,1.000002,1.000002,1.000002,1.000002,1.000002
2,TIC 23769326,1,0.999834,0.999834,0.999834,0.999834,0.999834,0.999834,0.999834,0.999834,...,0.999834,0.999834,0.999834,0.999834,0.999834,0.999834,0.999834,0.999834,0.999834,0.999834
3,TIC 4749946,0,1.000079,1.000079,1.000079,1.000079,1.000079,1.000079,1.000079,1.000079,...,1.000079,1.000079,1.000079,1.000079,1.000079,1.000079,1.000079,1.000079,1.000079,1.000079
4,TIC 139086171,1,0.999962,0.999962,0.999962,0.999962,0.999962,0.999962,0.999962,0.999962,...,0.999962,0.999962,0.999962,0.999962,0.999962,0.999962,0.999962,0.999962,0.999962,0.999962
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7172,TIC 330866902,0,0.999994,0.999994,0.999994,0.999994,0.999994,0.999994,0.999994,0.999994,...,0.999994,0.999994,0.999994,0.999994,0.999994,0.999994,0.999994,0.999994,0.999994,0.999994
7173,TIC 124543547,1,0.999937,0.999937,0.999937,0.999937,0.999937,0.999937,0.999937,0.999937,...,0.999937,0.999937,0.999937,0.999937,0.999937,0.999937,0.999937,0.999937,0.999937,0.999937
7174,TIC 354034141,0,0.999998,0.999998,0.999998,0.999998,0.999998,0.999998,0.999998,0.999998,...,0.999998,0.999998,0.999998,0.999998,0.999998,0.999998,0.999998,0.999998,0.999998,0.999998
7175,TIC 33690636,1,0.999996,0.999996,0.999996,0.999996,0.999996,0.999996,0.999996,0.999996,...,0.999996,0.999996,0.999996,0.999996,0.999996,0.999996,0.999996,0.999996,0.999996,0.999996


In [8]:
# star[['tid']].to_csv('targets-20240320-101.csv')

In [ ]:
# star_x = star.drop('confirmed_planet',axis=1)
# star_y = star[['confirmed_planet']]
# star_x = star_x.apply(lambda row: row.fillna(0), axis=1)

In [ ]:
# df_long = star_x.melt(id_vars=['tid'], var_name='time', value_name='value')
# df_long

In [ ]:
# it_list = df_long['tid'].unique()

In [ ]:
# extracted_features = pd.DataFrame()

In [ ]:
# for i in range(85,len(it_list)):
#     print('Index',i)
#     temp_feat = extract_features(df_long[df_long['tid']==it_list[i]], column_id='tid', column_sort='time')
#     extracted_features = pd.concat([extracted_features,temp_feat])

In [ ]:
# extracted_features = extracted_features.reset_index().rename(columns = {'index':'tid'}) 

In [ ]:
# extracted_features.to_csv('features.csv')

In [45]:
star_list = [int(i[0][4:]) for i in star[['tid']].to_numpy()]

In [64]:
from astroquery.mast import Catalogs

tic_id = star_list

catalog_data = Catalogs.query_criteria(catalog="Tic", ID=tic_id)


In [65]:
catalog_data = catalog_data[['ID','Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']]

In [66]:
catalog_data['ID'] = ['TIC ' + str(id) for id in catalog_data['ID']]

In [71]:
catalog_data = catalog_data.to_pandas()

In [72]:
catalog_data = catalog_data.rename(columns={'ID': 'tid'})
catalog_data

,tid,Teff,logg,MH,rad,mass,rho,lum,Tmag,ra,dec,plx
0,TIC 346929661,6178.00,4.33057,NaN,1.229390,1.180,0.635064,1.983640,10.19760,0.977583,59.334777,4.88382
1,TIC 432549364,6596.04,4.24670,0.131000,1.462160,1.376,0.440187,3.645990,10.22420,0.362154,39.383828,3.69704
2,TIC 347011522,6278.00,4.03312,NaN,1.767780,1.230,0.222649,4.373576,12.02050,1.186752,54.196718,1.48756
3,TIC 347013211,6125.85,4.02861,0.072000,1.723440,1.157,0.226019,3.768365,10.77990,1.119139,54.934463,2.69680
4,TIC 354594208,6033.55,4.31097,-0.196000,1.224510,1.119,0.609452,1.790248,9.76119,1.887827,32.437771,6.10335
...,...,...,...,...,...,...,...,...,...,...,...,...
6921,TIC 144440290,5709.00,4.38728,NaN,1.070770,1.020,0.830838,1.097297,8.76350,359.161504,-44.719078,11.87170
6922,TIC 183532609,5589.00,4.43663,0.240000,0.996641,0.990,1.000040,0.873190,9.14680,359.900297,-35.031368,11.08730
6923,TIC 184240683,5735.75,4.36856,0.154118,1.094100,1.020,0.778811,1.167261,11.54640,359.348986,-41.277152,3.20634
6924,TIC 201233610,5736.00,3.96944,NaN,1.732250,1.020,0.196230,2.926543,12.95280,359.516493,-49.300827,1.04193


In [73]:
new_star = pd.merge(star, catalog_data, on='tid', how='inner')

new_star

,tid,confirmed_planet,1,2,3,4,5,6,7,8,...,logg,MH,rad,mass,rho,lum,Tmag,ra,dec,plx
0,TIC 18067025,1,0.999847,0.999847,0.999847,0.999847,0.999847,0.999847,0.999847,0.999847,...,4.52282,NaN,0.795922,0.770,1.527140,0.296211,13.08740,184.759983,36.609666,3.11271
1,TIC 4749723,0,1.000002,1.000002,1.000002,1.000002,1.000002,1.000002,1.000002,1.000002,...,3.57778,-0.0290,2.670800,0.984,0.051650,6.216298,8.14936,188.826441,33.047742,6.72848
2,TIC 23769326,1,0.999834,0.999834,0.999834,0.999834,0.999834,0.999834,0.999834,0.999834,...,4.27706,NaN,1.318540,1.200,0.523483,2.355028,12.33420,208.944741,40.391802,1.62523
3,TIC 4749946,0,1.000079,1.000079,1.000079,1.000079,1.000079,1.000079,1.000079,1.000079,...,4.59371,0.2135,0.766162,0.840,1.867750,0.338409,9.75140,188.904944,35.027144,13.32070
4,TIC 139086171,1,0.999962,0.999962,0.999962,0.999962,0.999962,0.999962,0.999962,0.999962,...,4.47278,-0.0940,0.921586,0.920,1.175380,0.618640,11.30930,184.625038,34.898123,4.95353
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7172,TIC 330866902,0,0.999994,0.999994,0.999994,0.999994,0.999994,0.999994,0.999994,0.999994,...,4.47172,NaN,0.942560,0.960,1.146420,0.731361,10.62420,200.807306,34.396176,6.20585
7173,TIC 124543547,1,0.999937,0.999937,0.999937,0.999937,0.999937,0.999937,0.999937,0.999937,...,3.60916,NaN,2.686240,1.070,0.055201,7.850926,10.56720,104.480686,-9.924756,2.06101
7174,TIC 354034141,0,0.999998,0.999998,0.999998,0.999998,0.999998,0.999998,0.999998,0.999998,...,4.46160,NaN,1.016120,1.090,1.038940,1.176887,11.02100,189.300286,49.524392,4.18270
7175,TIC 33690636,1,0.999996,0.999996,0.999996,0.999996,0.999996,0.999996,0.999996,0.999996,...,4.34359,NaN,0.932128,0.699,0.863079,0.310287,11.04270,93.122352,-16.791933,8.03588


In [87]:
new_star = new_star.drop_duplicates(subset='tid', keep='first')

In [90]:
new_star = new_star.fillna(0)

In [92]:
new_star.to_csv('updated_database_exoplanet.csv')